# Computation and storage

Which nodes lie on shortest paths through the imported network? Project the
prior to NetworkX, compute betweenness, attach the result by node ID, and save
both a native graph and a Cytoscape exchange file.

Requires `annnet[polars,storage,networkx]` and the shared synthetic tutorial data.
The result describes this teaching topology, not a biological ranking.


In [1]:
from pathlib import Path
import annnet as an
import polars as pl

# Repository root, notebook directory, or a downloaded notebook beside data/.
DATA = next((p for p in (
    Path('docs/guide/data'), Path('data')
) if (p / 'interactions.csv').is_file()), None)
if DATA is None:
    raise FileNotFoundError('Download the tutorial CSV files into a data/ directory.')

interactions = pl.read_csv(DATA / 'interactions.csv')
G = an.from_edge_frame(interactions, edge_id='edge_id', sign='effect', slice='prior')
G.attrs.backend = 'polars'


## What an export represents

A backend graph is a projection of one AnnNet context. Binary graph libraries
cannot directly express every hyperedge, node-layer value, or named slice.
Choose the algorithm input before interpreting its output:

| Hyperedge mode | Representation and consequence |
| --- | --- |
| `skip` | Omits the relation from computation. |
| `reify` / `star` | Adds a relation node; paths and node statistics include it. |
| `expand` / `clique` | Adds pairwise connections; the pairs alone lose the original grouping. |

Adapters have defaults; NetworkX and igraph default to `skip`. Explicit
participant coefficients make expansion refuse by default unless you accept
loss with `coefficients='drop'`. Also decide whether to retain parallel edges,
how to aggregate them if not, and how to represent mixed direction or layers.
An export can support reconstruction while still being the wrong input to a
particular algorithm.

## Make the algorithm input explicit

Select the `prior` slice and state the hyperedge policy. This dataset contains
only binary edges, so `skip` removes none. If you later introduce hyperedges,
reconsider the projection. Keep the returned manifest for re-import.


In [2]:
import networkx as nx

nx_graph, manifest = an.to_nx(G, slice='prior', hyperedges='skip')
print(type(nx_graph).__name__, nx_graph.number_of_nodes(), nx_graph.number_of_edges())
assert nx_graph.number_of_edges() == 11


MultiDiGraph 10 11


The projected graph preserves the parallel evidence records. Different
algorithms have different support for multigraphs and directed graphs; establish
that the representation fits the method before interpreting its result.

Here `weight=None` counts hops. Neither confidence nor an inhibitory sign is a
path length. We leave both out of this calculation deliberately.


In [3]:
scores = nx.betweenness_centrality(nx_graph, weight=None)
G.attrs.update('nodes', {node: {'betweenness': value} for node, value in scores.items()})
G.uns['analysis'] = {
    'method': 'networkx.betweenness_centrality',
    'networkx_version': nx.__version__,
    'slice': 'prior',
    'weight': None,
    'hyperedges': 'skip',
}
G.attrs.table('nodes').sort('betweenness', descending=True)


node_id,betweenness
str,f64
"""RAS""",0.166667
"""RAF""",0.166667
"""SOS1""",0.138889
"""MEK""",0.138889
"""GRB2""",0.083333
"""ERK""",0.083333
"""AKT""",0.013889
"""EGFR""",0.0
"""DUSP6""",0.0


A higher score means a node participates in more shortest paths under these
settings. It does not estimate activity or establish importance in an experiment.
The update joins scores by node ID, while the graph metadata records how they
were obtained.


## Use a cached accessor when convenient

`G.nx` combines projection and dispatch. For this call, explicitly select the
same slice. Clear the accessor after edits before requesting a new projection:
current automatic invalidation does not cover every removal or attribute change.
Fresh `an.to_nx` calls avoid that cache.


In [4]:
G.nx.clear()
via_accessor = G.nx.betweenness_centrality(
    G, weight=None, _nx_slice='prior', _nx_hyperedge='skip'
)
assert all(abs(via_accessor[node] - value) < 1e-12 for node, value in scores.items())


## Save the annotated analysis and export for Cytoscape

The native object is the analysis artifact. CX2 is an exchange representation
for Cytoscape; `reify` makes any hyperedges explicit as relation nodes for that
consumer. AnnNet embeds reconstruction metadata in the CX2 output.

A temporary directory makes this example safe to rerun. Replace it with your
analysis directory when you want to keep the files.


In [5]:
from tempfile import TemporaryDirectory

with TemporaryDirectory(prefix='annnet-analysis-') as directory:
    directory = Path(directory)
    G.write(directory / 'analysis.annnet')
    an.to_cx2(G, directory / 'analysis.cx2', hyperedges='reify')
    restored = an.read(directory / 'analysis.annnet')
    print('Files:', sorted(path.name for path in directory.iterdir()))

assert set(restored.E) == set(G.E)
assert restored.attrs.row('nodes', 'ERK')['betweenness'] == scores['ERK']
assert restored.uns['analysis']['slice'] == 'prior'
restored.attrs.table('nodes').sort('betweenness', descending=True)


Files: ['analysis.annnet', 'analysis.cx2']


node_id,betweenness
str,f64
"""RAS""",0.166667
"""RAF""",0.166667
"""SOS1""",0.138889
"""MEK""",0.138889
"""GRB2""",0.083333
"""ERK""",0.083333
"""AKT""",0.013889
"""EGFR""",0.0
"""DUSP6""",0.0


For a backend round trip, keep both values returned by the adapter and use
`an.from_nx(nx_graph, manifest)`. For GraphML, CSV, and other formats that use a
sidecar, keep the companion file as well. An external format alone may not
retain every annotation, slice, or richer relationship.


## Persistence and reconstruction channels

Use the native `.annnet` format to resume work in AnnNet. JSON/NDJSON, Parquet,
and CX2 also keep reconstruction extensions inside their output. Runtime graph
adapters return a separate manifest; GraphML/GEXF, SIF, CSV, and Excel may write
a sidecar. Keep that metadata with the export. Sidecars contain a checksum and
are refused if they no longer match the main file.

SBML and PyG serve domain-specific tasks and have different limits. The format
comparison in the API reference details what each route can retain.

Attached measurement arrays are materialized only for existing node-layer
placements by the native writer. Use `attached='error'` to refuse that operation
or `attached='drop'` to omit the backings deliberately. Save an external assay
separately when it contains values outside those placements. The writer can
reconstruct matrices from structure records; `matrix=True` also persists the
matrix as a load-time/size tradeoff.

## History, snapshots, and reproducibility

History records some mutating calls, not every edit. Current hooks can miss
removals, attribute updates, and slice creation. Snapshots compare sets of
node, edge, and slice IDs. They do not capture a full revision or support replay.

The next example demonstrates the distinction: an attribute-only edit changes
the graph while leaving a membership diff empty; removing an edge changes that
diff. Retain the input files, analysis code, parameters, package versions, and
saved graph alongside any history export.

In [6]:
before = G.history.snapshot('before_annotation')
G.attrs.update('nodes', {'ERK': {'note': 'reviewed'}})
assert G.history.diff(before).is_empty()
G.remove_edges('egfr_grb2_alt')
difference = G.history.diff(before)
print(difference.summary())
assert difference.edges_removed == {'egfr_grb2_alt'}

Diff: before_annotation - current

Nodes: +0 added, 0 removed
Edges: +0 added, 1 removed
slices: +0 added, 0 removed


In [7]:
G.history.mark('review_complete')
with TemporaryDirectory(prefix='annnet-history-') as directory:
    event_count = G.history.export(str(Path(directory) / 'history.ndjson'))
print('Exported events:', event_count)

Exported events: 3


A marker gives the event log a useful checkpoint label, but does not fill gaps
in mutation coverage. A fresh adapter conversion or an explicit accessor clear
is also needed before recomputing on a graph edited after an earlier projection.

## API and related topics

[Backend accessors](../reference/core/backend-accessors.md) · [Format comparison](../reference/io/format-comparison.md) · [Native format](../reference/io/annnet-format.md) · [History API](../reference/core/history.md)

[Guide contents](index.md) · [Sample data](data/index.md)